# A child cannot enter the owner’s lifecycle

The standard wrapper does not let a child manage its owner’s connection.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The lifecycle code comes from AOA itself. The example supplies only the resource or external connection used to observe it.

In [ ]:
!pip install -q "aoa-action-machine[postgres]"

In [2]:
import asyncio

from aoa.action_machine.exceptions import TransactionProhibitedError
from aoa.action_machine.resources.postgres import PostgresPool, PostgresStorage
from aoa.action_machine.resources.sql import SqlStorageWrapper

## 2. Define the objects used in this case

The recording connection logs commands and rejects overlapping operations. The pool limits borrowers; neither class parses SQL or stores database rows.

In [3]:
class RecordingConnection:
    """Record commands; do not parse SQL or simulate database storage."""
    def __init__(self, number):
        """Give this connection a visible identity."""
        self.number = number
        self.commands = []
        self.busy = False
        self.closed = False
    async def execute(self, query, *params):
        """Reject concurrent use and record one command."""
        if self.busy or self.closed:
            raise RuntimeError("connection is busy or closed")
        self.busy = True
        try:
            await asyncio.sleep(0)
            self.commands.append(query)
            return "OK"
        finally:
            self.busy = False
    async def close(self):
        """Mark the physical connection closed."""
        self.closed = True

class RecordingPool:
    """Offer a fixed number of observable connections without a server."""
    def __init__(self, size=1):
        """Create the fixed capacity used by this example."""
        self.connections = [RecordingConnection(i + 1) for i in range(size)]
        self.free = asyncio.Queue()
        for connection in self.connections:
            self.free.put_nowait(connection)
        self.held = set()
        self.closed = False
    async def acquire(self, *, timeout=None):
        """Wait for an available connection with the supplied bound."""
        connection = await asyncio.wait_for(self.free.get(), timeout)
        self.held.add(connection)
        return connection
    async def release(self, connection):
        """Return a borrowed connection exactly once."""
        self.held.remove(connection)
        self.free.put_nowait(connection)
    async def close(self):
        """Close the pool only after all borrowers have returned."""
        assert not self.held
        for connection in self.connections:
            await connection.close()
        self.closed = True





class ExampleStorage(PostgresStorage):
    """A PostgreSQL resource that works through the example's pool."""


async def main():
    """Run this case and verify its observable result."""
    pool = RecordingPool()
    owner = ExampleStorage(pool=PostgresPool(pool=pool))
    await owner.open()
    child_view = SqlStorageWrapper(owner)
    try:
        async with child_view:
           raise AssertionError("The context must not open")
    except TransactionProhibitedError as error:
        print(type(error).__name__)
    else:
        raise AssertionError("The wrapper allowed lifecycle control")
    assert len(pool.held) == 1
    print("owner still holds connection:", len(pool.held) == 1)
    await owner.close()
    await pool.close()

## 3. Prepare this call

Create the state and declarations used below. Merely declaring a per-call factory does not run it.

## 4. Run and check the observation

The assertions test this case’s stated behavior. Printed output is recorded below; expected failures are caught explicitly.

In [4]:
await main()

TransactionProhibitedError
owner still holds connection: True


## 5. Explain the result

The refusal happens before the owner is touched. The parent can continue using the connection and remains responsible for ending its transaction. The outer scope later releases the resource.